In [ ]:
# ===== PPO BASELINE (Two-Hump replication) — CONFIG (edit ONLY this cell) ====
# Runtime: A100 GPU. Everything here is PyTorch; nothing in this notebook needs numba.

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH   = "experiments/ppo"     # branch holding experiments/ppo/
REPO_DIR = "ACSolverX"
CLONE       = True
UPDATE_REPO = True            # git reset --hard so a RESTART pulls the latest push
MOUNT_DRIVE = True            # checkpoints + jsonl mirrored to Drive if True

# `network.py` imports distrax, but the only thing it ever does with it is
# `distrax.Categorical(logits=...)` and read `.logits`. The real package depends
# on tfp-nightly -- a build that changes every day, landing next to the pinned
# TensorFlow already on the Colab image -- so installing it can break a runtime
# that was working, to gain one wrapper class. run_ppo ships a Categorical shim
# that is verified bit-identical to distrax 0.1.9 against the real 610model
# (3.7933267034162554e-05 from both), so the shim is the default. Set True to
# install the genuine package instead; the parity report says which one ran.
INSTALL_REAL_DISTRAX = False

# --- the ladder: each rung is cheaper than the next and gates it ------------
#   convert        orbax -> .npz once, so nothing after this needs orbax    (seconds)
#   parity         upstream weights in torch vs JAX, same batch, TF32 off   (minutes)
#   beam_upstream  beam-decode the SHIPPED 610model over the 1190 MS set.
#                  This is a replication with ZERO training: that artefact is
#                  step 1000 of 4376 and is named for ~610 solves, so expect
#                  roughly 605-610. A few off is consistent with the two
#                  documented beam deviations (64-bit dedup hashes, torch-seeded
#                  hash vector). Tens off means STOP -- something in the env,
#                  the net or the decode is wrong and training will not fix it.
#   train          PPO from scratch, one run per (arm, seed), resumable     (hours+)
#   beam_trained   beam-decode each trained checkpoint over the same 1190
#   report         no GPU work: aggregates the artefacts on disk into one
#                  smoke_report.json (GPU, parity verdict, solve rate, measured
#                  seconds/presentation, projected full-run hours, and a
#                  certificate check of every solved row)
#
# Start with the first three. Add "train"/"beam_trained" once beam_upstream lands
# near 610 -- that is the gate.
STAGES = ["convert", "parity", "beam_upstream"]

# --- 5-minute GPU smoke ------------------------------------------------------
# True  -> convert, parity, a TIME-BOUNDED slice of beam_upstream, then report.
#          Nothing trains. Overrides STAGES below.
#
# The beam runs at FULL production width (1024 x 150) and simply stops after
# SMOKE_SECONDS, between presentations. That matters: every row it writes is a
# real row at the real settings, so the full run RESUMES where the smoke stopped
# and keeps them -- no work is thrown away, and the measured seconds/presentation
# extrapolates directly to all 1190. A narrow-beam smoke would do neither.
#
# It writes smoke_report.json to Drive and prints it between ==== lines. Paste
# that back before starting the full run.
SMOKE_RUN     = True
SMOKE_SECONDS = 300

# --- experiment arms --------------------------------------------------------
# "1190MS"        -> paper row PPO-SUB-DRT         588.2 (585-591). All 1190 MS
#                    presentations pinned, 1190 more envs resampling from them.
# "AC19_extended" -> paper row PPO-SUB-DRT + AC-19 607.2 (605-610). Its first 634
#                    lines ARE the first 634 lines of 1190MS, which is exactly the
#                    `parallel_sample[:634] = False` pin in ppo_ac_s.py; the other
#                    156,128 rows are the AC-19 curriculum the sampling envs draw
#                    from. This is the file with the "634 solved presentations".
# Deliberately absent: AC-1M (too big) and raw data/AC19.txt -- that file pins
# NOTHING (0-length shared prefix; only 171 of the 1190 MS rows appear anywhere
# in it), so it is not the paper's "+ AC-19" arm. See the swap-in block below.
ARMS  = ["1190MS", "AC19_extended"]

# The paper's figure is a mean over 5 seeds. Run ONE seed first and read the
# `sps` in the heartbeat before deciding how many fit: [142, 1, 2, 3, 4].
SEEDS = [142]

# Upstream trained 4376 updates; the checkpoint it shipped is step 1000, which is
# the defensible session-bounded target. Training is resumable -- re-running this
# notebook continues from the checkpoint instead of restarting, so raising this
# later costs only the extra updates.
MAX_UPDATES = 1000

# --- output -----------------------------------------------------------------
LOCAL_OUT_DIR = "results/ppo"                                   # written here
DRIVE_OUT_DIR = "/content/drive/MyDrive/acsolverx_results/ppo"  # mirrored here

# --- runner knobs (upstream hyperparameters are NOT restated here) -----------
# LR / NUM_ENVS / NUM_STEPS / GAMMA / clip / entropy / ... all come from
# experiments/ppo/ppo.py:DEFAULT_CONFIG, which tests/ppo asserts field by field
# against the shipped 610model's own saved metadata. Restating them in a notebook
# is how they drift, so this cell only carries knobs that are ours.
cfg = {
    "DEVICE": "auto",

    # Upstream pins jax_default_matmul_precision=float32. An A100 silently uses
    # TF32 for fp32 matmuls unless told not to, which is ~1e-3 relative error --
    # fatal for the parity gate, invisible in training. `parity` forces it off
    # regardless; set True to buy the speed back in `train`.
    "ALLOW_TF32": False,

    "MICRO_BATCH": 2048,      # gradient accumulation: memory only, not the loss
    "ROLLOUT_CHUNK": 4096,    # forward-pass split during collection
    "SAVE_EVERY": 25,         # checkpoint + Drive mirror every N updates
    "HEARTBEAT_EVERY_S": 60,  # TIME-based progress line

    # the upstream checkpoint
    "CKPT_DIR":   "ppo_checkpoints/610model",
    "CKPT_STEP":  None,                             # None = newest step present
    "PARAMS_NPZ": "ppo_checkpoints/610model_params.npz",

    # --- evaluation: the ONLY thing that produces the paper's number --------
    # 1190MS is the paper's denominator. Training-time `num_solved` is NOT this
    # number -- on the AC-19 arm it counts over 156,762 rows.
    "EVAL_DATASET": "1190MS",
    "EVAL_START": 0,
    "EVAL_END": None,                 # None = all 1190
    "BEAM_WIDTH": 1024,               # beam/beam_search.py argparse defaults
    "BEAM_MAX_STEPS": 150,
    "BEAM_ALPHA": 0.0,                # value-head weight in the beam score
    "BEAM_TEMPERATURE": 0.0,          # >0 = Gumbel sampling; then SEED matters
    "BEAM_TEMP_END": 0.0,
    "BEAM_TIME_BUDGET_S": None,       # wall-clock stop BETWEEN presentations
    "PARITY_BATCH": 256,

    # ---- Weights & Biases (training only; none of it changes a result) -----
    "USE_WANDB": True,
    "AUTO_AUTHENTICATE_WANDB": False,  # True: promptless via Colab Secret WANDB_API_KEY
    "WANDB_ENTITY": "avigyapaudel045-aisc",
    "WANDB_PROJECT": "acsolver",
    "WANDB_GROUP": "ppo-replication",  # one group per batch of runs
    "WANDB_JOB_TYPE": "ppo-train",
    "WANDB_TAGS": ["ppo", "two-hump", "replication"],
    "WANDB_NOTES": None,
}

# --- swap-ins ---------------------------------------------------------------
# Full 5-seed replication once one seed's wall-clock is known:
#   SEEDS = [142, 1, 2, 3, 4]
#
# The full upstream schedule instead of the shipped-checkpoint step:
#   MAX_UPDATES = 4376
#
# Exercise the TRAINING path too in the smoke (two updates, meaningless numbers,
# but it proves the optimiser step, the checkpoint and the Drive mirror all work):
#   SMOKE_TRAIN = True   # read by the RUN cell
#
# Raw data/AC19.txt as a clearly-labelled EXTRA (not a paper row -- it pins no
# presentation, so its policy never sees an MS target deterministically):
#   ARMS = ["1190MS", "AC19_extended", "AC19"]

print("config loaded.")


In [ ]:
# ==================== SETUP (clone / pull / install / mount / auth) ========
import os, sys, subprocess

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)                       # anchor so re-runs never nest the clone
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    # torch and jax are PREINSTALLED on the Colab GPU image with matching CUDA
    # wheels -- reinstalling either is how a working runtime gets broken. Only
    # what is missing: flax (the JAX net the parity gate compares against),
    # orbax-checkpoint (reads ppo_checkpoints/610model), and wandb.
    sh("pip -q install flax orbax-checkpoint wandb")
    # distrax: OFF by default, and on its own line so a failure here stops
    # nothing. See INSTALL_REAL_DISTRAX in CONFIG for why the shim is preferred.
    if INSTALL_REAL_DISTRAX:
        sh("pip -q install distrax")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    # local: walk up from cwd to the repo root (dir holding experiments/ + data/)
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

# run from repo root so "data/..." and "import experiments..." resolve
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)

# The production beam budget (1024 x 150 = 153,600 expansions) is far above the
# repo's local cap; the cap exists so a laptop session cannot start a production
# search by accident. This notebook IS the production run, so it opts in.
os.environ["ACSOLVERX_ALLOW_BIG"] = "1"

# SETUP's `git reset --hard` rewrites the .py files on disk, but Python keeps the
# OLD module objects in sys.modules for the life of the runtime -- so the RUN
# cell's `from experiments.ppo.run_ppo import main` would silently reuse stale
# code (a pull is NOT a reload). Drop them so the next import reads what SETUP
# just fetched. Without this you must Runtime -> Restart session.
import importlib
for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

# ---- the runtime we actually got ----------------------------------------
if IN_COLAB:
    sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
import torch
print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()} "
      f"device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
if not torch.cuda.is_available():
    print("WARNING: no GPU. `train` on CPU is not viable; Runtime -> Change runtime type -> A100.")

# ---- W&B authentication -------------------------------------------------
# AUTO_AUTHENTICATE_WANDB (set in CONFIG):
#   True  -> PROMPTLESS auth from a Colab Secret named WANDB_API_KEY (persists
#            across runtime restarts). Create it ONCE: Colab left sidebar ->
#            key icon (Secrets) -> add WANDB_API_KEY = <your key> -> toggle
#            'Notebook access' ON. Without a secret it reuses a valid key from
#            this session, else prompts once (a stale/invalid key is discarded,
#            not reused -- so it can always recover).
#   False -> prompt for the API key EVERY run (use to switch/refresh a key).
if cfg["USE_WANDB"]:
    import re, wandb

    def _clean_key(k):
        # strip whitespace/newlines and stray surrounding quotes from a paste
        return (k or "").strip().strip('"').strip("'").strip()

    def _valid(k):
        return bool(re.fullmatch(r"[A-Za-z0-9_]+", k or ""))

    def _prompt_key():
        import getpass
        return _clean_key(getpass.getpass(
            "Paste your W&B API key (from wandb.ai/authorize), then Enter: "))

    _auto = cfg.get("AUTO_AUTHENTICATE_WANDB", True)
    if not _auto:
        os.environ["WANDB_API_KEY"] = _prompt_key()          # always ask for a fresh key
    else:
        # auto: prefer a Colab Secret; else reuse a VALID session key; else prompt.
        _key, _why = None, ""
        try:
            from google.colab import userdata
            _key = _clean_key(userdata.get("WANDB_API_KEY"))
        except Exception as _e:
            _why = type(_e).__name__   # SecretNotFoundError / NotebookAccessError
        if _key:
            os.environ["WANDB_API_KEY"] = _key
            print("W&B: using Colab Secret WANDB_API_KEY (promptless).")
        elif _valid(os.environ.get("WANDB_API_KEY", "")):
            print("W&B: reusing the key from this session.")
        else:
            if os.environ.get("WANDB_API_KEY"):     # stale/invalid -> drop it, don't reuse
                print("W&B: discarding an invalid key left in this session.")
                os.environ.pop("WANDB_API_KEY", None)
            print("W&B: no usable Colab Secret WANDB_API_KEY"
                  f"{(' (' + _why + ')') if _why else ''}.")
            print("     -> Promptless auth: Colab left sidebar -> key icon (Secrets)")
            print("        -> Add  name=WANDB_API_KEY  value=<key from wandb.ai/authorize>")
            print("        -> toggle 'Notebook access' ON, then re-run this cell.")
            print("     Pasting once for THIS session instead:")
            os.environ["WANDB_API_KEY"] = _prompt_key()

    # final format check before hitting the server (Colab getpass can mangle a paste)
    if not _valid(os.environ.get("WANDB_API_KEY", "")):
        print("W&B: that key still has invalid characters (allowed: A-Z a-z 0-9 _).")
        print("     Re-copy it EXACTLY from https://wandb.ai/authorize.")

    # verify; relogin=True overwrites any stale key cached in ~/.netrc
    try:
        wandb.login(key=os.environ.get("WANDB_API_KEY", ""), relogin=True, verify=True)
        try:
            _default = wandb.Api().default_entity
        except Exception:
            _default = None
        _target = cfg["WANDB_ENTITY"] or _default
        print(f"W&B: authenticated ✓  runs -> {_target}/{cfg['WANDB_PROJECT']}")
    except Exception as e:
        print(f"W&B: authentication FAILED ✗ -- {e}")
        print("     tip: add a Colab Secret WANDB_API_KEY (promptless), or set "
              "AUTO_AUTHENTICATE_WANDB=False and re-run to paste a fresh key.")

In [ ]:
# ==================== RUN ================================================
# Every stage reads what is already on disk before doing anything, so
# Restart -> Run All continues: `train` resumes from its checkpoint and
# `beam_eval` skips presentations already in its jsonl.
import os
from experiments.ppo.ppo import make_config
from experiments.ppo.run_ppo import main, train_tag

# jsonl and checkpoints are APPENDED locally and mirrored to Drive whole-file --
# never append to a mount.
cfg["OUT_DIR"] = os.path.join(REPO_ROOT, LOCAL_OUT_DIR)
cfg["MIRROR_DIR"] = DRIVE_OUT_DIR if (IN_COLAB and MOUNT_DRIVE) else None
cfg["MAX_UPDATES"] = MAX_UPDATES

# The smoke overrides the ladder rather than being a separate code path: it runs
# the SAME stages with the SAME beam settings, bounded in wall-clock. Keeping the
# width at production is what makes its timing extrapolate and its rows reusable.
if SMOKE_RUN:
    SMOKE_TRAIN = globals().get("SMOKE_TRAIN", False)
    STAGES = ["convert", "parity", "beam_upstream"]
    if SMOKE_TRAIN:
        STAGES += ["train", "beam_trained"]
        ARMS, SEEDS, MAX_UPDATES = ARMS[:1], SEEDS[:1], 2
        cfg["MAX_UPDATES"] = MAX_UPDATES
    cfg["BEAM_TIME_BUDGET_S"] = SMOKE_SECONDS
    cfg["EVAL_END"] = None      # the time budget stops it, not a row count
    cfg["USE_WANDB"] = False
    cfg["SAVE_EVERY"] = 1

# The report is disk-only -- no GPU, no cost -- so EVERY ladder ends with one,
# not just the smoke. Without this, flipping SMOKE_RUN to False would decode all
# 1190 presentations and then print no table to read them from.
if "report" not in STAGES:
    STAGES += ["report"]

if SMOKE_RUN:
    print(f"SMOKE_RUN: {' -> '.join(STAGES)}; beam bounded to {SMOKE_SECONDS}s at "
          f"width {cfg['BEAM_WIDTH']} x {cfg['BEAM_MAX_STEPS']} (production settings)")
else:
    print(f"FULL RUN: {' -> '.join(STAGES)}; no time budget, width "
          f"{cfg['BEAM_WIDTH']} x {cfg['BEAM_MAX_STEPS']} over all of "
          f"{cfg['EVAL_DATASET']}; rows already on disk or in Drive are skipped")

BASE = dict(make_config())
BASE.update(cfg)

def run(stage, **over):
    c = dict(BASE)
    c.update(over)
    c["STAGE"] = stage
    return main(c)

results = {}

if "convert" in STAGES:
    results["convert"] = run("convert")

if "parity" in STAGES:
    results["parity"] = run("parity", DATASET=ARMS[0])

if "beam_upstream" in STAGES:
    # BEAM_CHECKPOINT=None -> the transplanted upstream weights (the 610model).
    results["beam_upstream"] = run("beam_eval", BEAM_CHECKPOINT=None)

for arm in ARMS:
    for seed in SEEDS:
        tag = train_tag(arm, seed)
        if "train" in STAGES:
            results["train:" + tag] = run("train", DATASET=arm, SEED=seed)
        if "beam_trained" in STAGES:
            ckpt = os.path.join(cfg["OUT_DIR"], tag + ".pt")
            if not os.path.exists(ckpt):
                print(f"skip beam_trained for {tag}: no checkpoint yet "
                      f"(add \"train\" to STAGES)")
                continue
            results["beam_trained:" + tag] = run("beam_eval", BEAM_CHECKPOINT=ckpt)

# Last, so it sees every artefact the run produced. No GPU work -- it only reads
# what is on disk, which is also why re-running just this stage is free.
if "report" in STAGES:
    results["report"] = run("report", SMOKE_RUN=SMOKE_RUN)

print("\ndone:", ", ".join(results))


In [ ]:
# ==================== TABLE ==============================================
# A fourth cell because the table has a different lifetime from the run: it is
# the deliverable, it aggregates ACROSS sessions (seeding missing jsonls back
# from the Drive mirror first), and re-printing it must never re-run a stage.
# It reads beam jsonls over the 1190 MS presentations and nothing else -- a
# training log's `num_solved` counts a different denominator and is refused.
from experiments.ppo.results_table import print_table

_ = print_table(cfg["OUT_DIR"], eval_stem=cfg["EVAL_DATASET"],
                mirror_dir=cfg["MIRROR_DIR"])
